# Route A: bacpipe BirdNET

T4 recommended. Runtime, Run all. First run downloads BirdNET weights.

Uses bacpipe's built-in test wavs, then this package (PCA, UMAP, trajectory, change-point, HMM).

bacpipe requires Python &lt; 3.13. Colab is 3.13, so the install cell uses `--no-deps` and keeps Colab TF/Torch.

## 1. Clone

In [ ]:
REPO = "https://github.com/ST-48-1240162/bioacoustic-embedding-dynamics.git"

%cd /content
!rm -rf bioacoustic-embedding-dynamics
!git clone --depth 1 {REPO}
%cd bioacoustic-embedding-dynamics

## 2. Install

In [ ]:
import sys
!{sys.executable} -m pip install --ignore-requires-python --no-deps bacpipe
!{sys.executable} -m pip install -q umap-learn ruptures hmmlearn
!{sys.executable} -m pip install -q --no-deps -e .

import librosa
if not hasattr(librosa, "get_duration"):
    from librosa.core.audio import get_duration as _gd
    librosa.get_duration = _gd

## 3. BirdNET embeddings

In [ ]:
from pathlib import Path

import bacpipe
import torch

MODEL = "birdnet"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
audio_dir = Path(bacpipe.__file__).parent / "tests" / "test_data"

bacpipe.config.models = [MODEL]
bacpipe.config.dashboard = False
bacpipe.config.audio_dir = str(audio_dir)
bacpipe.settings.device = DEVICE

print("device:", DEVICE)
print("audio_dir:", audio_dir)
bacpipe.ensure_models_exist(model_names=[MODEL])
loader = bacpipe.generate_embeddings(
    model_name=MODEL,
    audio_dir=str(audio_dir),
    check_if_already_processed=True,
)
print("embedding_size:", loader.metadata_dict.get("embedding_size"))

## 4. Manifest and analysis

In [ ]:
from pathlib import Path

from bioacoustic_embedding_dynamics.adapters import bacpipe_loader_to_manifest

MANIFEST = Path("data/bacpipe_birdnet.jsonl")
bacpipe_loader_to_manifest({MODEL: loader}, MODEL, MANIFEST, min_confidence=0.0)
print("lines:", sum(1 for _ in MANIFEST.open()))

In [ ]:
!python -m bioacoustic_embedding_dynamics.cli --manifest {MANIFEST} --out reports/bacpipe --seed 42

## 5. Summary and figures

In [ ]:
import json
from pathlib import Path
from IPython.display import Image, display

summary = json.loads(Path("reports/bacpipe/summary.json").read_text())
print(json.dumps(summary, indent=2))

for name in [
    "pca_species.png", "umap_species.png", "trajectory_pca.png",
    "changepoints.png", "trajectory_changepoints.png", "hmm_regimes.png", "shuffle_null.png",
]:
    display(Image(filename=str(Path("reports/bacpipe") / name)))